# Gold (Athena + Glue Catalog)

Construção da camada analítica a partir das 4 tabelas Silver e do stream
de eventos, usando CTAS no Athena. Resultado: 5 tabelas Gold particionadas
em Parquet/SNAPPY, com 13 testes de qualidade.

## 1. Configuração e catalogação das tabelas Silver

As tabelas Silver estavam no S3 mas não no Glue Catalog — sem catálogo,
o Athena não as enxerga. Optamos por `wr.s3.store_parquet_metadata()` em vez
de Glue Crawler: custo zero (Crawler cobra por execução, mínimo 10 min),
tipos lidos do footer do Parquet em vez de inferidos por amostragem, e
schema versionado no repositório em vez de gerado fora do controle.


In [1]:
import awswrangler as wr
import pandas as pd
import boto3

BUCKET = "brazil-literacy-lakehouse-joaopaulo"
DATABASE = "brazil_literacy_lakehouse"
ATHENA_OUTPUT = f"s3://{BUCKET}/athena-results/"

boto3.setup_default_session(region_name="sa-east-1")

wr.catalog.databases()

,Database,Description
0,brazil_literacy_lakehouse,


In [2]:
wr.catalog.tables(database=DATABASE)

,Database,Table,Description,TableType,Columns,Partitions
0,brazil_literacy_lakehouse,eventos_indicador_municipio,,EXTERNAL_TABLE,"evento_id, tipo_evento, timestamp_evento, ano,...",
1,brazil_literacy_lakehouse,gold_indicador_municipio,,EXTERNAL_TABLE,"id_municipio, sigla_uf, rede, taxa_alfabetizac...",ano
2,brazil_literacy_lakehouse,gold_meta_vs_resultado,,EXTERNAL_TABLE,"id_municipio, sigla_uf, rede, meta, resultado,...",ano
3,brazil_literacy_lakehouse,silver_alunos,,EXTERNAL_TABLE,"id_municipio, id_escola, id_aluno, caderno, se...",ano
4,brazil_literacy_lakehouse,silver_brasil,,EXTERNAL_TABLE,"rede, taxa_alfabetizacao, meta_alfabetizacao_2...",ano
5,brazil_literacy_lakehouse,silver_municipio,,EXTERNAL_TABLE,"id_municipio, serie, rede, taxa_alfabetizacao,...",ano
6,brazil_literacy_lakehouse,silver_uf,,EXTERNAL_TABLE,"sigla_uf, serie, rede, taxa_alfabetizacao, med...",ano


In [3]:
res_uf = wr.s3.store_parquet_metadata(
    path=f"s3://{BUCKET}/silver/uf/",
    database=DATABASE,
    table="silver_uf",
    dataset=True,
    mode="overwrite",
)

res_uf

({'sigla_uf': 'string',
  'serie': 'string',
  'rede': 'string',
  'taxa_alfabetizacao': 'double',
  'media_portugues': 'double',
  'proporcao_aluno_nivel_0': 'double',
  'proporcao_aluno_nivel_1': 'double',
  'proporcao_aluno_nivel_2': 'double',
  'proporcao_aluno_nivel_3': 'double',
  'proporcao_aluno_nivel_4': 'double',
  'proporcao_aluno_nivel_5': 'double',
  'proporcao_aluno_nivel_6': 'double',
  'proporcao_aluno_nivel_7': 'double',
  'proporcao_aluno_nivel_8': 'double',
  'meta_alfabetizacao_2024': 'double',
  'meta_alfabetizacao_2025': 'double',
  'meta_alfabetizacao_2026': 'double',
  'meta_alfabetizacao_2027': 'double',
  'meta_alfabetizacao_2028': 'double',
  'meta_alfabetizacao_2029': 'double',
  'meta_alfabetizacao_2030': 'double',
  'percentual_participacao': 'double',
  'taxa_alfabetizacao_divergente': 'boolean'},
 {'ano': 'string'},
 {'s3://brazil-literacy-lakehouse-joaopaulo/silver/uf/ano=2023/': ['2023'],
  's3://brazil-literacy-lakehouse-joaopaulo/silver/uf/ano=2024/'

In [4]:
df_teste = wr.athena.read_sql_query(
    "SELECT count(*) AS linhas FROM silver_uf",
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

df_teste

,linhas
0,145


In [5]:
tabelas_silver = ["municipio", "brasil", "alunos"]

for t in tabelas_silver:
    res = wr.s3.store_parquet_metadata(
        path=f"s3://{BUCKET}/silver/{t}/",
        database=DATABASE,
        table=f"silver_{t}",
        dataset=True,
        mode="overwrite",
    )
    print(f"silver_{t}: {len(res[0])} colunas, {len(res[2])} partições")

silver_municipio: 25 colunas, 2 partições
silver_brasil: 10 colunas, 3 partições
silver_alunos: 12 colunas, 2 partições


In [6]:
df_valida = wr.athena.read_sql_query(
    """
    SELECT 'municipio' AS tabela, count(*) AS linhas FROM silver_municipio
    UNION ALL SELECT 'brasil', count(*) FROM silver_brasil
    UNION ALL SELECT 'alunos', count(*) FROM silver_alunos
    UNION ALL SELECT 'uf', count(*) FROM silver_uf
    """,
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

df_valida

,tabela,linhas
0,municipio,23995
1,uf,145
2,brasil,3
3,alunos,3867999


## 2. Helper de criação idempotente

O CTAS do Athena falha se a tabela já existe ou se o caminho S3 tem arquivos.
`criar_gold()` resolve: drop no catálogo → limpeza do S3 → CTAS → validação
de contagem. Parquet + SNAPPY + particionamento por decisão de FinOps
(Athena cobra por dado escaneado).


In [4]:
def criar_gold(nome_tabela, sql_select, particao=None):
    """Cria (ou recria) uma tabela Gold via CTAS, de forma idempotente."""
    caminho = f"s3://{BUCKET}/gold/{nome_tabela}/"

    wr.catalog.delete_table_if_exists(database=DATABASE, table=nome_tabela)

    wr.s3.delete_objects(caminho)

    props = [
        f"external_location = '{caminho}'",
        "format = 'PARQUET'",
        "parquet_compression = 'SNAPPY'",
    ]
    if particao:
        props.append(f"partitioned_by = ARRAY{particao}")

    ddl = f"""
    CREATE TABLE {nome_tabela}
    WITH ({', '.join(props)})
    AS
    {sql_select}
    """

    wr.athena.start_query_execution(
        sql=ddl,
        database=DATABASE,
        s3_output=ATHENA_OUTPUT,
        wait=True,
    )

    total = wr.athena.read_sql_query(
        f"SELECT count(*) AS linhas FROM {nome_tabela}",
        database=DATABASE,
        s3_output=ATHENA_OUTPUT,
    )
    print(f"{nome_tabela}: {total['linhas'][0]:,} linhas → {caminho}")
    return total

## 3. gold_indicador_municipio (23.995 linhas)

Indicador por município. A coluna `faixa_alfabetizacao` testa a regra de
faixas de 10 pontos decifrada na Fase 4 — confirmada em 10.584 linhas, com
34 exceções de fronteira explicadas por arredondamento em 2 casas decimais.
Identificado o município 2305100 (CE, 2023) com incoerência interna entre
`taxa_alfabetizacao` e `nivel_alfabetizacao`.


In [9]:
sql_indicador = """
SELECT
    id_municipio,
    sigla_uf,
    rede,
    taxa_alfabetizacao,
    nivel_alfabetizacao,
    CASE
        WHEN taxa_alfabetizacao < 40 THEN 'Muito baixo (<40%)'
        WHEN taxa_alfabetizacao < 50 THEN 'Baixo (40-50%)'
        WHEN taxa_alfabetizacao < 60 THEN 'Medio-baixo (50-60%)'
        WHEN taxa_alfabetizacao < 70 THEN 'Medio (60-70%)'
        WHEN taxa_alfabetizacao < 80 THEN 'Alto (70-80%)'
        ELSE 'Muito alto (>=80%)'
    END AS faixa_alfabetizacao,
    media_portugues,
    percentual_participacao,
    taxa_alfabetizacao_divergente,
    ano
FROM silver_municipio
"""

criar_gold("gold_indicador_municipio", sql_indicador, particao=["ano"])

gold_indicador_municipio: 23,995 linhas → s3://brazil-literacy-lakehouse-joaopaulo/gold/gold_indicador_municipio/


,linhas
0,23995


In [10]:
wr.athena.read_sql_query(
    """
    SELECT nivel_alfabetizacao, faixa_alfabetizacao, count(*) AS linhas,
           min(taxa_alfabetizacao) AS min_taxa,
           max(taxa_alfabetizacao) AS max_taxa
    FROM gold_indicador_municipio
    WHERE nivel_alfabetizacao IS NOT NULL
    GROUP BY nivel_alfabetizacao, faixa_alfabetizacao
    ORDER BY nivel_alfabetizacao
    """,
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

,nivel_alfabetizacao,faixa_alfabetizacao,linhas,min_taxa,max_taxa
0,0,Muito baixo (<40%),1599,4.35,39.99
1,0,Baixo (40-50%),6,40.00,40.00
2,1,Medio-baixo (50-60%),11,50.00,50.00
3,1,Baixo (40-50%),1418,40.00,49.97
4,2,Medio-baixo (50-60%),1778,50.00,59.99
5,2,Muito alto (>=80%),1,91.67,91.67
6,2,Medio (60-70%),4,60.00,60.00
7,3,Alto (70-80%),5,70.00,70.00
8,3,Medio (60-70%),1893,60.00,69.98
9,4,Alto (70-80%),1802,70.00,79.99


In [11]:
wr.athena.read_sql_query(
    """
    SELECT id_municipio, sigla_uf, ano, rede,
           taxa_alfabetizacao, nivel_alfabetizacao,
           media_portugues, percentual_participacao,
           taxa_alfabetizacao_divergente
    FROM gold_indicador_municipio
    WHERE nivel_alfabetizacao = 2 AND taxa_alfabetizacao > 80
    """,
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

,id_municipio,sigla_uf,ano,rede,taxa_alfabetizacao,nivel_alfabetizacao,media_portugues,percentual_participacao,taxa_alfabetizacao_divergente
0,2305100,CE,2023,Municipal,91.67,2,811.44,98.61,True


In [12]:
wr.athena.read_sql_query(
    """
    SELECT id_municipio, sigla_uf, ano, rede,
           taxa_alfabetizacao, nivel_alfabetizacao, media_portugues
    FROM gold_indicador_municipio
    WHERE taxa_alfabetizacao_divergente = true
    ORDER BY id_municipio
    """,
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

,id_municipio,sigla_uf,ano,rede,taxa_alfabetizacao,nivel_alfabetizacao,media_portugues
0,2304905,CE,2023,Municipal,99.15,5,838.52
1,2305100,CE,2023,Municipal,91.67,2,811.44
2,5106752,MT,2023,Municipal,65.94,3,759.14


## 4. gold_meta_vs_resultado (36.624 linhas)

Unpivot das 7 colunas largas de meta (2024-2030) via `CROSS JOIN UNNEST`.
Verificado antes que a meta é atributo fixo do município (0 divergências
entre anos em 5.232 combinações), exigindo `DISTINCT` para não duplicar.
Achado: 53,3% dos municípios atingiram a meta em 2024; os piores gaps têm
participação alta (81-100%) e média de proficiência entre 704 e 727 — logo
abaixo do corte de 743, evidenciando a sensibilidade do indicador ao limiar.


In [13]:
wr.athena.read_sql_query(
    """
    SELECT
        count(*) AS combinacoes,
        count_if(metas_distintas > 1) AS com_meta_divergente
    FROM (
        SELECT id_municipio, rede,
               count(DISTINCT meta_alfabetizacao_2024) AS metas_distintas,
               count(*) AS linhas
        FROM silver_municipio
        WHERE meta_alfabetizacao_2024 IS NOT NULL
        GROUP BY id_municipio, rede
    )
    """,
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

,combinacoes,com_meta_divergente
0,5232,0


In [14]:
sql_meta = """
WITH metas AS (
    SELECT DISTINCT
        id_municipio, sigla_uf, rede,
        meta_alfabetizacao_2024, meta_alfabetizacao_2025, meta_alfabetizacao_2026,
        meta_alfabetizacao_2027, meta_alfabetizacao_2028, meta_alfabetizacao_2029,
        meta_alfabetizacao_2030
    FROM silver_municipio
    WHERE meta_alfabetizacao_2024 IS NOT NULL
),
metas_long AS (
    SELECT id_municipio, sigla_uf, rede, ano_meta, meta
    FROM metas
    CROSS JOIN UNNEST(
        ARRAY['2024','2025','2026','2027','2028','2029','2030'],
        ARRAY[meta_alfabetizacao_2024, meta_alfabetizacao_2025, meta_alfabetizacao_2026,
              meta_alfabetizacao_2027, meta_alfabetizacao_2028, meta_alfabetizacao_2029,
              meta_alfabetizacao_2030]
    ) AS t(ano_meta, meta)
),
resultados AS (
    SELECT id_municipio, rede, ano, taxa_alfabetizacao
    FROM silver_municipio
)
SELECT
    m.id_municipio,
    m.sigla_uf,
    m.rede,
    m.meta,
    r.taxa_alfabetizacao AS resultado,
    CASE WHEN r.taxa_alfabetizacao IS NOT NULL
         THEN round(r.taxa_alfabetizacao - m.meta, 2) END AS gap,
    CASE WHEN r.taxa_alfabetizacao IS NULL THEN NULL
         WHEN r.taxa_alfabetizacao >= m.meta THEN true
         ELSE false END AS meta_atingida,
    m.ano_meta AS ano
FROM metas_long m
LEFT JOIN resultados r
    ON  r.id_municipio = m.id_municipio
    AND r.rede = m.rede
    AND r.ano = m.ano_meta
"""

criar_gold("gold_meta_vs_resultado", sql_meta, particao=["ano"])

gold_meta_vs_resultado: 36,624 linhas → s3://brazil-literacy-lakehouse-joaopaulo/gold/gold_meta_vs_resultado/


,linhas
0,36624


In [15]:
wr.athena.read_sql_query(
    """
    SELECT
        count(*) AS linhas_2024,
        round(avg(gap), 2) AS gap_medio,
        round(min(gap), 2) AS pior_gap,
        round(max(gap), 2) AS melhor_gap,
        count_if(meta_atingida) AS atingiram,
        round(100.0 * count_if(meta_atingida) / count(*), 1) AS pct_atingiram
    FROM gold_meta_vs_resultado
    WHERE ano = '2024'
    """,
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

,linhas_2024,gap_medio,pior_gap,melhor_gap,atingiram,pct_atingiram
0,5232,1.11,-68.9,84.27,2788,53.3


In [16]:
wr.athena.read_sql_query(
    """
    SELECT id_municipio, sigla_uf, rede, meta, resultado, gap
    FROM gold_meta_vs_resultado
    WHERE ano = '2024'
    ORDER BY gap ASC
    LIMIT 5
    """,
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

,id_municipio,sigla_uf,rede,meta,resultado,gap
0,4320453,RS,Municipal,80.00,11.1,-68.90
1,4301073,RS,Municipal,80.00,18.2,-61.80
2,4310850,RS,Municipal,80.00,25.0,-55.00
3,4319752,RS,Municipal,80.00,25.0,-55.00
4,4319208,RS,Municipal,61.56,7.1,-54.46


In [17]:
wr.athena.read_sql_query(
    """
    SELECT g.id_municipio, g.sigla_uf, g.gap, g.resultado,
           i.percentual_participacao, i.media_portugues
    FROM gold_meta_vs_resultado g
    JOIN gold_indicador_municipio i
      ON  i.id_municipio = g.id_municipio
      AND i.rede = g.rede
      AND i.ano = g.ano
    WHERE g.ano = '2024'
    ORDER BY g.gap ASC
    LIMIT 10
    """,
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

,id_municipio,sigla_uf,gap,resultado,percentual_participacao,media_portugues
0,4320453,RS,-68.90,11.10,100.00,712.1694
1,4301073,RS,-61.80,18.20,84.62,713.5351
2,4319752,RS,-55.00,25.00,100.00,723.1819
3,4310850,RS,-55.00,25.00,100.00,713.7748
4,4319208,RS,-54.46,7.10,82.35,704.4793
5,4306700,RS,-53.30,26.70,93.75,718.8282
6,4302584,RS,-52.70,27.30,100.00,727.0937
7,4313334,RS,-52.00,28.00,96.15,714.9629
8,1713601,TO,-51.85,19.71,81.01,718.7000
9,4320354,RS,-51.27,23.50,85.71,709.4142


## 5. gold_evolucao_temporal (12.650 linhas)

Pivot de 2023/2024 por município e rede. Separa variação em pontos
percentuais de variação relativa (confusão comum). 6.491 municípios
melhoraram, 4.826 pioraram, 1.305 sem comparativo.


In [5]:
sql_evolucao = """
WITH pivot AS (
    SELECT
        id_municipio,
        sigla_uf,
        rede,
        max(CASE WHEN ano = '2023' THEN taxa_alfabetizacao END) AS taxa_2023,
        max(CASE WHEN ano = '2024' THEN taxa_alfabetizacao END) AS taxa_2024,
        max(CASE WHEN ano = '2023' THEN media_portugues END) AS media_portugues_2023,
        max(CASE WHEN ano = '2024' THEN media_portugues END) AS media_portugues_2024
    FROM gold_indicador_municipio
    GROUP BY id_municipio, sigla_uf, rede
)
SELECT
    id_municipio,
    sigla_uf,
    rede,
    taxa_2023,
    taxa_2024,
    round(taxa_2024 - taxa_2023, 2) AS variacao_pp,
    CASE
        WHEN taxa_2023 IS NULL OR taxa_2024 IS NULL THEN NULL
        WHEN taxa_2023 = 0 THEN NULL
        ELSE round(100.0 * (taxa_2024 - taxa_2023) / taxa_2023, 2)
    END AS variacao_percentual,
    CASE
        WHEN taxa_2023 IS NULL OR taxa_2024 IS NULL THEN 'Sem comparativo'
        WHEN taxa_2024 > taxa_2023 THEN 'Melhorou'
        WHEN taxa_2024 < taxa_2023 THEN 'Piorou'
        ELSE 'Estavel'
    END AS tendencia,
    media_portugues_2023,
    media_portugues_2024,
    round(media_portugues_2024 - media_portugues_2023, 2) AS variacao_media_portugues
FROM pivot
"""

criar_gold("gold_evolucao_temporal", sql_evolucao)

gold_evolucao_temporal: 12,650 linhas → s3://brazil-literacy-lakehouse-joaopaulo/gold/gold_evolucao_temporal/


,linhas
0,12650


In [6]:
wr.athena.read_sql_query(
    """
    SELECT tendencia,
           count(*) AS municipios,
           round(avg(variacao_pp), 2) AS variacao_media_pp,
           round(avg(variacao_media_portugues), 2) AS variacao_media_prof
    FROM gold_evolucao_temporal
    GROUP BY tendencia
    ORDER BY municipios DESC
    """,
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

,tendencia,municipios,variacao_media_pp,variacao_media_prof
0,Melhorou,6491,13.19,12.31
1,Piorou,4826,-12.43,-11.32
2,Sem comparativo,1305,NaN,NaN
3,Estavel,28,0.00,-3.08


## 6. gold_proficiencia_alunos (12.417 linhas)

Recálculo do indicador a partir dos 3,8M de registros de aluno.
Confirmado empiricamente o corte de 743 pontos. Reconciliação com o
indicador publicado: erro médio de 0,036 pp em 12.408 combinações.
A média ponderada por `peso_aluno` erra 19x menos que a simples —
evidência de que a fonte usa peso amostral.


In [8]:
wr.athena.read_sql_query(
    """
    SELECT
        alfabetizado,
        count(*) AS alunos,
        round(min(proficiencia), 2) AS min_prof,
        round(max(proficiencia), 2) AS max_prof,
        round(avg(proficiencia), 2) AS media_prof
    FROM silver_alunos
    WHERE proficiencia IS NOT NULL
    GROUP BY alfabetizado
    ORDER BY alfabetizado
    """,
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

,alfabetizado,alunos,min_prof,max_prof,media_prof
0,0,1370115,578.46,743.00,702.82
1,1,1984546,743.00,904.38,779.84


In [9]:
sql_prof = """
SELECT
    id_municipio,
    sigla_uf,
    rede,
    count(*) AS alunos_avaliados,
    count(DISTINCT id_escola) AS escolas,
    round(avg(proficiencia), 2) AS proficiencia_media,
    round(sum(proficiencia * peso_aluno) / sum(peso_aluno), 2) AS proficiencia_media_ponderada,
    round(approx_percentile(proficiencia, 0.5), 2) AS proficiencia_mediana,
    count_if(alfabetizado = '1') AS alunos_alfabetizados,
    round(100.0 * count_if(alfabetizado = '1') / count(*), 2) AS taxa_alfabetizacao_calculada,
    round(100.0 * sum(CASE WHEN alfabetizado = '1' THEN peso_aluno ELSE 0 END) / sum(peso_aluno), 2) AS taxa_alfabetizacao_ponderada,
    ano
FROM silver_alunos
WHERE proficiencia IS NOT NULL
  AND peso_aluno IS NOT NULL
GROUP BY id_municipio, sigla_uf, rede, ano
"""

criar_gold("gold_proficiencia_alunos", sql_prof, particao=["ano"])

gold_proficiencia_alunos: 12,417 linhas → s3://brazil-literacy-lakehouse-joaopaulo/gold/gold_proficiencia_alunos/


,linhas
0,12417


In [10]:
wr.athena.read_sql_query(
    """
    SELECT
        count(*) AS combinacoes,
        round(avg(abs(p.taxa_alfabetizacao_ponderada - i.taxa_alfabetizacao)), 3) AS erro_medio_abs,
        round(max(abs(p.taxa_alfabetizacao_ponderada - i.taxa_alfabetizacao)), 2) AS erro_max,
        count_if(abs(p.taxa_alfabetizacao_ponderada - i.taxa_alfabetizacao) <= 0.01) AS batem_exato,
        count_if(abs(p.taxa_alfabetizacao_ponderada - i.taxa_alfabetizacao) > 1) AS divergem_acima_1pp,
        round(avg(abs(p.taxa_alfabetizacao_calculada - i.taxa_alfabetizacao)), 3) AS erro_medio_simples
    FROM gold_proficiencia_alunos p
    JOIN gold_indicador_municipio i
      ON  i.id_municipio = p.id_municipio
      AND i.rede = p.rede
      AND i.ano = p.ano
    """,
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

,combinacoes,erro_medio_abs,erro_max,batem_exato,divergem_acima_1pp,erro_medio_simples
0,12408,0.036,36.06,9273,56,0.675


In [11]:
wr.athena.read_sql_query(
    """
    SELECT p.id_municipio, p.sigla_uf, p.rede, p.ano,
           p.alunos_avaliados,
           p.taxa_alfabetizacao_ponderada AS calculada,
           i.taxa_alfabetizacao AS publicada,
           round(p.taxa_alfabetizacao_ponderada - i.taxa_alfabetizacao, 2) AS diferenca,
           i.taxa_alfabetizacao_divergente AS flag_fase4
    FROM gold_proficiencia_alunos p
    JOIN gold_indicador_municipio i
      ON  i.id_municipio = p.id_municipio
      AND i.rede = p.rede
      AND i.ano = p.ano
    WHERE abs(p.taxa_alfabetizacao_ponderada - i.taxa_alfabetizacao) > 1
    ORDER BY abs(p.taxa_alfabetizacao_ponderada - i.taxa_alfabetizacao) DESC
    LIMIT 10
    """,
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

,id_municipio,sigla_uf,rede,ano,alunos_avaliados,calculada,publicada,diferenca,flag_fase4
0,2305100,CE,Municipal,2023,71,55.61,91.67,-36.06,True
1,5105622,MT,Estadual,2023,171,46.45,19.85,26.60,False
2,5103908,MT,Estadual,2024,29,36.67,59.62,-22.95,False
3,5107040,MT,Estadual,2024,23,56.52,33.91,22.61,False
4,2506905,PB,Estadual,2024,12,33.33,50.57,-17.24,False
5,1304005,AM,Municipal,2023,106,52.54,35.75,16.79,False
6,2304905,CE,Municipal,2023,118,83.11,99.15,-16.04,True
7,2506806,PB,Municipal,2024,128,47.24,37.97,9.27,False
8,1300607,AM,Municipal,2023,429,44.00,38.02,5.98,False
9,4308300,RS,Municipal,2023,43,58.14,62.05,-3.91,False


## 7. gold_indicador_streaming (23.995 linhas)

Materialização do estado atual a partir do fluxo de eventos, com
deduplicação por evento mais recente (`row_number()`). Reconciliação
batch × streaming: 23.995 combinações idênticas, 0 órfãs, diferença 0 —
prova de consistência da ingestão híbrida.


In [12]:
wr.athena.read_sql_query(
    """
    SELECT count(*) AS eventos,
           count(DISTINCT evento_id) AS eventos_unicos,
           count(DISTINCT id_municipio) AS municipios,
           min(timestamp_evento) AS primeiro,
           max(timestamp_evento) AS ultimo,
           array_agg(DISTINCT rede ORDER BY rede) AS redes,
           array_agg(DISTINCT tipo_evento) AS tipos
    FROM eventos_indicador_municipio
    """,
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

,eventos,eventos_unicos,municipios,primeiro,ultimo,redes,tipos
0,23995,23995,5550,2026-08-28T05:25:21.596256+00:00,2026-08-28T05:25:22.339130+00:00,"[0, 2, 3, 5]",[nova_medicao_indicador]


### Primeira tentativa — bug de rótulo de `rede`

Os rótulos de `rede` foram redigitados no `CASE` em vez de puxados da
mesma lógica de `padronizar_rede_codigo()` (Fase 4), gerando divergência
de texto ("Publica" sem acento, "Total" abreviado). A reconciliação
acusou: 34.859 combinações batidas em vez de 23.995, com 21.728 órfãs —
terceira ocorrência da mesma classe de bug no projeto (nomenclatura
redigitada manualmente em vez de vinda de uma fonte única de verdade).
Corrigido na célula seguinte.


In [14]:
sql_stream = """
WITH ranked AS (
    SELECT
        id_municipio,
        ano,
        CASE rede
            WHEN 0 THEN 'Total'
            WHEN 2 THEN 'Estadual'
            WHEN 3 THEN 'Municipal'
            WHEN 5 THEN 'Publica (Estadual e Municipal)'
            ELSE CAST(rede AS varchar)
        END AS rede,
        taxa_alfabetizacao,
        evento_id,
        CAST(from_iso8601_timestamp(timestamp_evento) AS timestamp) AS ts_evento,
        fonte,
        row_number() OVER (
            PARTITION BY id_municipio, ano, rede
            ORDER BY from_iso8601_timestamp(timestamp_evento) DESC, evento_id DESC
        ) AS rn
    FROM eventos_indicador_municipio
)
SELECT
    id_municipio,
    CAST(ano AS varchar) AS ano_referencia,
    rede,
    taxa_alfabetizacao AS taxa_streaming,
    evento_id AS ultimo_evento_id,
    ts_evento AS ultimo_evento_em,
    fonte
FROM ranked
WHERE rn = 1
"""

criar_gold("gold_indicador_streaming", sql_stream)

gold_indicador_streaming: 23,995 linhas → s3://brazil-literacy-lakehouse-joaopaulo/gold/gold_indicador_streaming/


,linhas
0,23995


In [15]:
wr.athena.read_sql_query(
    """
    SELECT
        count(*) AS combinacoes_casadas,
        count_if(abs(s.taxa_streaming - b.taxa_alfabetizacao) < 0.001) AS identicas,
        round(max(abs(s.taxa_streaming - b.taxa_alfabetizacao)), 4) AS maior_diferenca,
        (SELECT count(*) FROM gold_indicador_streaming) AS total_streaming,
        (SELECT count(*) FROM gold_indicador_municipio) AS total_batch
    FROM gold_indicador_streaming s
    FULL OUTER JOIN gold_indicador_municipio b
      ON  b.id_municipio = s.id_municipio
      AND b.ano = s.ano_referencia
      AND b.rede = s.rede
    """,
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

,combinacoes_casadas,identicas,maior_diferenca,total_streaming,total_batch
0,34859,13131,0.0,23995,23995


In [16]:
wr.athena.read_sql_query(
    """
    SELECT 'streaming' AS origem, rede, count(*) AS linhas
    FROM gold_indicador_streaming GROUP BY rede
    UNION ALL
    SELECT 'batch', rede, count(*)
    FROM gold_indicador_municipio GROUP BY rede
    ORDER BY rede, origem
    """,
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

,origem,rede,linhas
0,batch,Estadual,2235
1,streaming,Estadual,2235
2,batch,Municipal,10896
3,streaming,Municipal,10896
4,streaming,Publica (Estadual e Municipal),10466
5,batch,Pública (Estadual e Municipal),10466
6,streaming,Total,398
7,batch,"Total (Federal, Estadual, Municipal e Privada)",398


### Correção — rótulos alinhados ao dicionário oficial

Refeito com os mesmos rótulos de `padronizar_rede_codigo()` (Fase 4).
Reconciliação após a correção: 23.995 combinações casadas, 0 órfãs,
diferença 0 — confirma que o problema era só de nomenclatura, não de dado.


In [17]:
sql_stream = """
WITH ranked AS (
    SELECT
        id_municipio,
        ano,
        CASE rede
            WHEN 0 THEN 'Total (Federal, Estadual, Municipal e Privada)'
            WHEN 2 THEN 'Estadual'
            WHEN 3 THEN 'Municipal'
            WHEN 5 THEN 'Pública (Estadual e Municipal)'
            ELSE CAST(rede AS varchar)
        END AS rede,
        taxa_alfabetizacao,
        evento_id,
        CAST(from_iso8601_timestamp(timestamp_evento) AS timestamp) AS ts_evento,
        fonte,
        row_number() OVER (
            PARTITION BY id_municipio, ano, rede
            ORDER BY from_iso8601_timestamp(timestamp_evento) DESC, evento_id DESC
        ) AS rn
    FROM eventos_indicador_municipio
)
SELECT
    id_municipio,
    CAST(ano AS varchar) AS ano_referencia,
    rede,
    taxa_alfabetizacao AS taxa_streaming,
    evento_id AS ultimo_evento_id,
    ts_evento AS ultimo_evento_em,
    fonte
FROM ranked
WHERE rn = 1
"""

criar_gold("gold_indicador_streaming", sql_stream)

gold_indicador_streaming: 23,995 linhas → s3://brazil-literacy-lakehouse-joaopaulo/gold/gold_indicador_streaming/


,linhas
0,23995


In [18]:
wr.athena.read_sql_query(
    """
    SELECT
        count(*) AS combinacoes_casadas,
        count_if(abs(s.taxa_streaming - b.taxa_alfabetizacao) < 0.001) AS identicas,
        round(max(abs(s.taxa_streaming - b.taxa_alfabetizacao)), 4) AS maior_diferenca,
        count_if(s.id_municipio IS NULL) AS orfas_batch,
        count_if(b.id_municipio IS NULL) AS orfas_streaming
    FROM gold_indicador_streaming s
    FULL OUTER JOIN gold_indicador_municipio b
      ON  b.id_municipio = s.id_municipio
      AND b.ano = s.ano_referencia
      AND b.rede = s.rede
    """,
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)

,combinacoes_casadas,identicas,maior_diferenca,orfas_batch,orfas_streaming
0,23995,23995,0.0,0,0


## 8. Validação de qualidade — 13 testes

Duplicidade, nulos em chaves, sanidade de domínio e integridade
referencial entre as tabelas Gold. 13/13 aprovados.


In [19]:
def validar(nome, sql, esperado=0, comparador="igual"):
    """Roda um teste de qualidade. Espera que a query retorne uma coluna 'valor'."""
    r = wr.athena.read_sql_query(sql, database=DATABASE, s3_output=ATHENA_OUTPUT)
    valor = r["valor"][0]
    ok = (valor == esperado) if comparador == "igual" else (valor <= esperado)
    print(f"[{'PASS' if ok else 'FALHA'}] {nome}: {valor:,} (esperado {'≤' if comparador!='igual' else '='} {esperado:,})")
    return ok

resultados = []

# --- 1. Duplicidade nas chaves de negócio ---
resultados.append(validar(
    "Duplicidade em gold_indicador_municipio",
    """SELECT count(*) AS valor FROM (
         SELECT id_municipio, ano, rede FROM gold_indicador_municipio
         GROUP BY id_municipio, ano, rede HAVING count(*) > 1)"""))

resultados.append(validar(
    "Duplicidade em gold_meta_vs_resultado",
    """SELECT count(*) AS valor FROM (
         SELECT id_municipio, ano, rede FROM gold_meta_vs_resultado
         GROUP BY id_municipio, ano, rede HAVING count(*) > 1)"""))

resultados.append(validar(
    "Duplicidade em gold_evolucao_temporal",
    """SELECT count(*) AS valor FROM (
         SELECT id_municipio, rede FROM gold_evolucao_temporal
         GROUP BY id_municipio, rede HAVING count(*) > 1)"""))

resultados.append(validar(
    "Duplicidade em gold_proficiencia_alunos",
    """SELECT count(*) AS valor FROM (
         SELECT id_municipio, ano, rede FROM gold_proficiencia_alunos
         GROUP BY id_municipio, ano, rede HAVING count(*) > 1)"""))

resultados.append(validar(
    "Duplicidade em gold_indicador_streaming",
    """SELECT count(*) AS valor FROM (
         SELECT id_municipio, ano_referencia, rede FROM gold_indicador_streaming
         GROUP BY id_municipio, ano_referencia, rede HAVING count(*) > 1)"""))

print()

# --- 2. Nulos em chaves ---
resultados.append(validar(
    "Nulos em chave (gold_indicador_municipio)",
    """SELECT count(*) AS valor FROM gold_indicador_municipio
       WHERE id_municipio IS NULL OR ano IS NULL OR rede IS NULL
          OR sigla_uf IS NULL OR taxa_alfabetizacao IS NULL"""))

resultados.append(validar(
    "Nulos em chave (gold_meta_vs_resultado)",
    """SELECT count(*) AS valor FROM gold_meta_vs_resultado
       WHERE id_municipio IS NULL OR ano IS NULL OR rede IS NULL OR meta IS NULL"""))

print()

# --- 3. Sanidade de domínio ---
resultados.append(validar(
    "Taxas fora de 0-100 (indicador)",
    """SELECT count(*) AS valor FROM gold_indicador_municipio
       WHERE taxa_alfabetizacao < 0 OR taxa_alfabetizacao > 100"""))

resultados.append(validar(
    "Gaps fora de -100..100 (meta_vs_resultado)",
    """SELECT count(*) AS valor FROM gold_meta_vs_resultado
       WHERE gap IS NOT NULL AND (gap < -100 OR gap > 100)"""))

resultados.append(validar(
    "Proficiencia fora de 0-1000 (alunos)",
    """SELECT count(*) AS valor FROM gold_proficiencia_alunos
       WHERE proficiencia_media < 0 OR proficiencia_media > 1000"""))

resultados.append(validar(
    "Codigo IBGE fora de 7 digitos",
    """SELECT count(*) AS valor FROM gold_indicador_municipio
       WHERE length(id_municipio) <> 7"""))

print()

# --- 4. Integridade referencial entre Gold ---
resultados.append(validar(
    "Municipios em meta_vs_resultado sem match no indicador",
    """SELECT count(DISTINCT m.id_municipio) AS valor
       FROM gold_meta_vs_resultado m
       LEFT JOIN gold_indicador_municipio i ON i.id_municipio = m.id_municipio
       WHERE i.id_municipio IS NULL"""))

resultados.append(validar(
    "Divergencia batch x streaming",
    """SELECT count(*) AS valor
       FROM gold_indicador_streaming s
       FULL OUTER JOIN gold_indicador_municipio b
         ON b.id_municipio = s.id_municipio AND b.ano = s.ano_referencia AND b.rede = s.rede
       WHERE s.id_municipio IS NULL OR b.id_municipio IS NULL
          OR abs(s.taxa_streaming - b.taxa_alfabetizacao) >= 0.001"""))

print()
print("=" * 60)
print(f"RESULTADO: {sum(resultados)}/{len(resultados)} testes passaram")
print("=" * 60)

[PASS] Duplicidade em gold_indicador_municipio: 0 (esperado = 0)
[PASS] Duplicidade em gold_meta_vs_resultado: 0 (esperado = 0)
[PASS] Duplicidade em gold_evolucao_temporal: 0 (esperado = 0)
[PASS] Duplicidade em gold_proficiencia_alunos: 0 (esperado = 0)
[PASS] Duplicidade em gold_indicador_streaming: 0 (esperado = 0)

[PASS] Nulos em chave (gold_indicador_municipio): 0 (esperado = 0)
[PASS] Nulos em chave (gold_meta_vs_resultado): 0 (esperado = 0)

[PASS] Taxas fora de 0-100 (indicador): 0 (esperado = 0)
[PASS] Gaps fora de -100..100 (meta_vs_resultado): 0 (esperado = 0)
[PASS] Proficiencia fora de 0-1000 (alunos): 0 (esperado = 0)
[PASS] Codigo IBGE fora de 7 digitos: 0 (esperado = 0)

[PASS] Municipios em meta_vs_resultado sem match no indicador: 0 (esperado = 0)
[PASS] Divergencia batch x streaming: 0 (esperado = 0)

RESULTADO: 13/13 testes passaram
